# Vision Transformer (ViT) from Scratch

**Lecture 2, Part 4 — Vizuara Modern Robot Learning Bootcamp**

We build a **Vision Transformer** from scratch and see why it replaced CNNs for robot vision:

| Component | What it does | Key idea |
|-----------|-------------|----------|
| Patch Embedding | Split image into 16×16 patches → vectors | Images as sequences of "visual words" |
| Positional Encoding | Tell the model where each patch is | Spatial layout preserved |
| Transformer Blocks | Self-attention over patches | Global reasoning from layer 1 |
| CLS Token | Aggregate patch information | Single image-level vector |
| Projection Bridge | Align ViT dim → LLM dim | Connect vision encoder to language model |
| VLM Assembly | Visual tokens + text tokens → one sequence | Cross-modal attention |

By the end, you'll understand the full pipeline from raw pixels to a Vision-Language Model, and why this makes our mini-VLA's TinyCNN obsolete.

---

In [ ]:
!pip install -q torch torchvision numpy matplotlib Pillow 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from PIL import Image
import math

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor'] = BG
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Step 1: Create a Robot Workspace Image

We'll create a simple synthetic scene — a tabletop with colored objects — to trace through the entire ViT pipeline.

In [ ]:
def create_robot_scene(size=224):
    """Create a synthetic robot workspace image with colored objects."""
    img = np.ones((size, size, 3), dtype=np.float32) * 0.85  # light gray table

    # Brown table edge (bottom third)
    img[size*2//3:, :] = [0.55, 0.35, 0.2]

    # Red cup (circle)
    cy, cx, r = size//3, size//4, size//10
    yy, xx = np.ogrid[:size, :size]
    mask = (xx - cx)**2 + (yy - cy)**2 < r**2
    img[mask] = [0.85, 0.2, 0.15]

    # Blue plate (ellipse)
    cy2, cx2 = size//3, size*3//4
    mask2 = ((xx - cx2)/1.5)**2 + ((yy - cy2)/1.0)**2 < (r*0.8)**2
    img[mask2] = [0.2, 0.4, 0.75]

    # Green cube
    s = size // 12
    y0, x0 = size//2 - s//2, size//2 - s//2
    img[y0:y0+s, x0:x0+s] = [0.2, 0.65, 0.35]

    # Robot gripper (gray arm from top)
    gw = size // 20
    gx = size * 3 // 5
    img[:size//4, gx-gw:gx+gw] = [0.4, 0.4, 0.45]  # arm
    # Gripper tips
    img[size//4-5:size//4+5, gx-gw*2:gx-gw] = [0.35, 0.35, 0.4]
    img[size//4-5:size//4+5, gx+gw:gx+gw*2] = [0.35, 0.35, 0.4]

    return np.clip(img, 0, 1)

scene = create_robot_scene(224)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(scene)
ax.set_title('Robot Workspace (224×224)', fontsize=14, color=ACCENT, fontweight='bold')
ax.axis('off')

# Label objects
ax.annotate('Red Cup', (56, 50), fontsize=11, color='white', fontweight='bold',
            bbox=dict(boxstyle='round', fc=ACCENT, alpha=0.8))
ax.annotate('Blue Plate', (148, 50), fontsize=11, color='white', fontweight='bold',
            bbox=dict(boxstyle='round', fc=BLUE, alpha=0.8))
ax.annotate('Green Cube', (95, 125), fontsize=11, color='white', fontweight='bold',
            bbox=dict(boxstyle='round', fc=TEAL, alpha=0.8))
ax.annotate('Gripper', (140, 30), fontsize=11, color='white', fontweight='bold',
            bbox=dict(boxstyle='round', fc='gray', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"Image shape: {scene.shape}")
print(f"Total pixels: {224 * 224:,} = 50,176")
print(f"A Transformer can't process 50K pixels as individual tokens — too expensive!")
print(f"Solution: split into patches.")

---
## Step 2: Patch Embedding — Images as Sequences

The key ViT insight: **split the image into fixed-size patches**, flatten each patch into a vector, and treat them as "visual words" in a sequence.

For a 224×224 image with 16×16 patches:
- 224 / 16 = 14 patches per side
- 14 × 14 = **196 patch tokens**
- Each patch = 16 × 16 × 3 = 768 raw dimensions

In [ ]:
PATCH_SIZE = 16
IMG_SIZE = 224
N_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2  # 14×14 = 196
PATCH_DIM = PATCH_SIZE * PATCH_SIZE * 3     # 16×16×3 = 768

print(f"Image: {IMG_SIZE}×{IMG_SIZE}")
print(f"Patch size: {PATCH_SIZE}×{PATCH_SIZE}")
print(f"Grid: {IMG_SIZE//PATCH_SIZE}×{IMG_SIZE//PATCH_SIZE} = {N_PATCHES} patches")
print(f"Each patch: {PATCH_SIZE}×{PATCH_SIZE}×3 = {PATCH_DIM} raw dimensions")

In [ ]:
# Visualize: the image split into patches
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: original image with grid overlay
ax = axes[0]
ax.imshow(scene)
for i in range(0, IMG_SIZE + 1, PATCH_SIZE):
    ax.axhline(i, color=ACCENT, linewidth=0.5, alpha=0.6)
    ax.axvline(i, color=ACCENT, linewidth=0.5, alpha=0.6)
ax.set_title(f'Image with {IMG_SIZE//PATCH_SIZE}×{IMG_SIZE//PATCH_SIZE} Patch Grid',
             fontsize=13, color=ACCENT, fontweight='bold')
ax.axis('off')

# Right: patches laid out as a sequence
ax = axes[1]
n_show = 14  # show first row of patches
patches_row = []
for j in range(n_show):
    patch = scene[0:PATCH_SIZE, j*PATCH_SIZE:(j+1)*PATCH_SIZE]
    patches_row.append(patch)

# Show as horizontal strip with separators
strip = np.ones((PATCH_SIZE + 4, n_show * (PATCH_SIZE + 2) + 2, 3), dtype=np.float32) * 0.95
for j, p in enumerate(patches_row):
    x_start = j * (PATCH_SIZE + 2) + 2
    strip[2:2+PATCH_SIZE, x_start:x_start+PATCH_SIZE] = p

ax.imshow(strip)
ax.set_title(f'First Row → 14 Patches → "Visual Words"',
             fontsize=13, color=TEAL, fontweight='bold')
ax.axis('off')
for j in range(n_show):
    x_center = j * (PATCH_SIZE + 2) + 2 + PATCH_SIZE / 2
    ax.text(x_center, PATCH_SIZE + 3.5, f'{j}', ha='center', fontsize=8, color=ACCENT)

plt.suptitle('Step 1: Split Image into Patches',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Each patch is a {PATCH_SIZE}×{PATCH_SIZE} crop of the image.")
print(f"14 rows × 14 columns = {N_PATCHES} patches total.")
print(f"These become the 'tokens' for the Transformer — just like words in a sentence!")

In [ ]:
class PatchEmbedding(nn.Module):
    """Split image into patches and project each to d_model dimensions.

    Equivalent to: unfold into patches → flatten → linear projection.
    In practice, a single Conv2d with kernel_size=stride=patch_size does this.
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=768):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        # Conv2d with kernel=stride=patch_size: each output pixel = one patch
        self.proj = nn.Conv2d(in_channels, d_model,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: [batch, 3, H, W]
        x = self.proj(x)          # [batch, d_model, H/P, W/P]
        x = x.flatten(2)          # [batch, d_model, n_patches]
        x = x.transpose(1, 2)     # [batch, n_patches, d_model]
        return x


D_MODEL = 768  # standard ViT-Base dimension
patch_embed = PatchEmbedding(IMG_SIZE, PATCH_SIZE, 3, D_MODEL)

# Convert image to tensor: [1, 3, 224, 224]
img_tensor = torch.tensor(scene).permute(2, 0, 1).unsqueeze(0)  # [1, 3, 224, 224]

with torch.no_grad():
    patches = patch_embed(img_tensor)  # [1, 196, 768]

print(f"Input:  {list(img_tensor.shape)}  — 1 image, 3 channels, 224×224")
print(f"Output: {list(patches.shape)}  — 1 image, {N_PATCHES} patches, {D_MODEL}-d each")
print(f"\nThe Conv2d trick: kernel_size=stride={PATCH_SIZE}")
print(f"  → Each 'pixel' in the output = one non-overlapping patch")
print(f"  → Equivalent to: cut patches → flatten → linear projection")

n_params = sum(p.numel() for p in patch_embed.parameters())
print(f"\nParameters: {n_params:,} ({PATCH_DIM} × {D_MODEL} + {D_MODEL} bias)")

---
## Step 3: CLS Token + Positional Encoding

Two additions before the Transformer:

1. **CLS token** — a learnable token prepended to the sequence. After the Transformer, it aggregates information from all patches into a single image-level representation.
2. **Positional encoding** — so the model knows *where* each patch is in the 2D grid.

In [ ]:
# CLS token: a learnable vector prepended to the patch sequence
cls_token = nn.Parameter(torch.randn(1, 1, D_MODEL) * 0.02)

# Positional encoding: learnable (ViT uses learned, not sinusoidal)
# 196 patches + 1 CLS = 197 positions
pos_embed = nn.Parameter(torch.randn(1, N_PATCHES + 1, D_MODEL) * 0.02)

with torch.no_grad():
    B = patches.shape[0]
    # Prepend CLS token
    cls_tokens = cls_token.expand(B, -1, -1)  # [1, 1, 768]
    x = torch.cat([cls_tokens, patches], dim=1)  # [1, 197, 768]
    # Add positional encoding
    x = x + pos_embed  # [1, 197, 768]

print(f"After prepending CLS token:")
print(f"  Patches: {list(patches.shape)} → With CLS: {list(x.shape)}")
print(f"  Position 0: CLS token (aggregator)")
print(f"  Positions 1-196: patch tokens")
print(f"\nPositional encoding shape: {list(pos_embed.shape)}")
print(f"  197 learned position vectors — the model learns 2D spatial layout")

In [ ]:
# Visualize: what does positional encoding look like after training?
# We simulate a trained ViT's positional encoding by making nearby patches similar

# Create 2D positional encoding (learned ViTs discover grid structure)
grid_size = IMG_SIZE // PATCH_SIZE  # 14

# Cosine similarity between position embeddings
pe = pos_embed[0, 1:].detach()  # skip CLS, [196, 768]
pe_norm = pe / pe.norm(dim=-1, keepdim=True)
pe_sim = (pe_norm @ pe_norm.T).numpy()

# Pick a few reference patches and show their similarity to all others
ref_patches = [
    (3, 3, 'Top-left patch'),
    (7, 7, 'Center patch'),
    (3, 10, 'Top-right area'),
    (10, 7, 'Bottom-center'),
]

fig, axes = plt.subplots(1, len(ref_patches), figsize=(18, 4))
colors_list = [ACCENT, BLUE, TEAL, PURPLE]

for ax, (ry, rx, label), c in zip(axes, ref_patches, colors_list):
    ref_idx = ry * grid_size + rx
    sim_map = pe_sim[ref_idx].reshape(grid_size, grid_size)
    im = ax.imshow(sim_map, cmap='RdYlGn', vmin=-0.3, vmax=0.3)
    ax.plot(rx, ry, 'X', color='black', markersize=12, markeredgewidth=2)
    ax.set_title(f'{label}\n(row={ry}, col={rx})', fontsize=10, color=c, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Positional Encoding Similarity (random init — trained ViTs learn grid structure)',
             fontsize=13, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("X marks the reference patch. Color = cosine similarity to that patch.")
print("After training, nearby patches become similar — the ViT discovers 2D layout.")
print("This is remarkable: position embeddings are 1D (just an index), but the model")
print("learns they encode a 2D grid purely from the data.")

---
## Step 4: The Full Vision Transformer

Now we assemble everything: patch embedding → CLS + position → N Transformer blocks → output.

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, d_model, d_head):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_head, bias=False)
        self.W_k = nn.Linear(d_model, d_head, bias=False)
        self.W_v = nn.Linear(d_model, d_head, bias=False)
        self.d_head = d_head

    def forward(self, x):
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        weights = F.softmax(scores, dim=-1)
        return weights @ V, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        d_head = d_model // n_heads
        self.heads = nn.ModuleList([AttentionHead(d_model, d_head) for _ in range(n_heads)])
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        outs, weights = zip(*[h(x) for h in self.heads])
        return self.proj(torch.cat(outs, dim=-1)), list(weights)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, weights = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, weights


class VisionTransformer(nn.Module):
    """ViT: image → patch tokens → Transformer → per-patch + CLS outputs."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 d_model=768, n_heads=12, n_layers=12, d_ff=3072):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        n_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, d_model) * 0.02)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                           # [B, 196, d_model]
        cls = self.cls_token.expand(B, -1, -1)            # [B, 1, d_model]
        x = torch.cat([cls, x], dim=1)                    # [B, 197, d_model]
        x = x + self.pos_embed                            # add position info

        all_weights = []
        for block in self.blocks:
            x, weights = block(x)
            all_weights.append(weights)

        x = self.norm(x)
        return x, all_weights  # x[:, 0] = CLS, x[:, 1:] = patch tokens


# Build a SMALL ViT for this demo (ViT-Tiny: 6 layers, 384-d, 6 heads)
vit = VisionTransformer(
    img_size=224, patch_size=16, in_channels=3,
    d_model=384, n_heads=6, n_layers=6, d_ff=1536,
)

total_params = sum(p.numel() for p in vit.parameters())
print(f"ViT-Tiny (our demo model):")
print(f"  Layers: 6, d_model: 384, heads: 6, d_ff: 1536")
print(f"  Parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"\nFor comparison:")
print(f"  ViT-Base:  86M params  (12 layers, d=768, 12 heads)")
print(f"  ViT-Large: 307M params (24 layers, d=1024, 16 heads)")
print(f"  Our tiny:  {total_params/1e6:.1f}M params (6 layers, d=384, 6 heads)")

In [ ]:
# Run the ViT on our robot scene
with torch.no_grad():
    output, all_attn_weights = vit(img_tensor)  # output: [1, 197, 384]

cls_output = output[0, 0]      # [384] — image-level representation
patch_outputs = output[0, 1:]  # [196, 384] — per-patch representations

print(f"Input:  {list(img_tensor.shape)}  — 224×224 RGB image")
print(f"Output: {list(output.shape)}")
print(f"  CLS token: {list(cls_output.shape)}  — single vector for the whole image")
print(f"  Patch tokens: {list(patch_outputs.shape)}  — one 384-d vector per patch")
print(f"\nCompare to our TinyCNN from Part 2:")
print(f"  TinyCNN output: [batch, 128]  — one vector, no spatial info")
print(f"  ViT output: [batch, 196, 384]  — rich spatial features preserved!")

---
## Step 5: Visualize ViT Attention

Which patches does each head attend to? We can reshape the attention weights back to the 14×14 grid to create attention maps.

In [ ]:
# Extract attention weights from the last layer
# all_attn_weights[layer][head] → [197, 197]
last_layer_weights = all_attn_weights[-1]  # 6 heads

# CLS token attention: which patches does CLS attend to?
fig, axes = plt.subplots(1, 6, figsize=(22, 3.5))
head_colors = [ACCENT, BLUE, TEAL, PURPLE, WARM, '#666666']

for h_idx, (ax, weights) in enumerate(zip(axes, last_layer_weights)):
    # CLS token is row 0, attending to patches 1-196
    cls_attn = weights[0, 1:].detach().numpy()  # [196]
    attn_map = cls_attn.reshape(14, 14)

    # Overlay on image
    ax.imshow(scene)
    ax.imshow(np.kron(attn_map, np.ones((PATCH_SIZE, PATCH_SIZE))),
              alpha=0.6, cmap='hot', vmin=0)
    ax.set_title(f'Head {h_idx+1}', fontsize=11, color=head_colors[h_idx], fontweight='bold')
    ax.axis('off')

plt.suptitle('CLS Token Attention: Which Patches Does the CLS Token Look At? (Last Layer)',
             fontsize=14, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each head learns different attention patterns:")
print("  After training, some heads attend to objects, others to edges/textures.")
print("  Right now (random weights), patterns are meaningless.")
print("  But the STRUCTURE is ready — training fills it with meaning.")

In [ ]:
# Patch-to-patch attention: pick the RED CUP patch and see what it attends to
# Red cup is roughly at row=4, col=3 in the 14×14 grid (patch index = 4*14+3 = 59)
cup_row, cup_col = 4, 3
cup_patch_idx = cup_row * 14 + cup_col + 1  # +1 because CLS is at index 0

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Show attention from the cup patch across 4 heads
for h_idx, ax in enumerate(zip(axes)):
    ax = axes[h_idx]
    weights = last_layer_weights[h_idx]
    cup_attn = weights[cup_patch_idx, 1:].detach().numpy()  # skip CLS
    attn_map = cup_attn.reshape(14, 14)

    ax.imshow(scene)
    ax.imshow(np.kron(attn_map, np.ones((PATCH_SIZE, PATCH_SIZE))),
              alpha=0.6, cmap='hot', vmin=0)
    # Mark the source patch
    rect = Rectangle((cup_col * PATCH_SIZE, cup_row * PATCH_SIZE),
                      PATCH_SIZE, PATCH_SIZE,
                      linewidth=2, edgecolor='cyan', facecolor='none')
    ax.add_patch(rect)
    ax.set_title(f'Head {h_idx+1}', fontsize=11, color=head_colors[h_idx], fontweight='bold')
    ax.axis('off')

plt.suptitle('Red Cup Patch Attention: What Does the Cup Look At? (Cyan = source)',
             fontsize=14, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("After training on real images, the cup patch would attend to:")
print("  • Other parts of the cup (handle, rim) — part-whole understanding")
print("  • The gripper — is it approaching?")
print("  • The table surface — where to place it?")
print("\nThis is the GLOBAL reasoning CNNs can't do:")
print("  CNN: cup sees only its 3×3 neighbors")
print("  ViT: cup sees the ENTIRE scene in one attention layer")

---
## Step 6: CNN vs ViT — Side by Side

Our TinyCNN from Part 2 used 3 convolution layers with 3×3 kernels. Let's compare what each architecture "sees".

In [ ]:
# Build the same TinyCNN from Part 2 (Build_MiniVLA notebook)
class TinyCNN(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.proj = nn.Linear(128, d_model)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.mean(dim=[2, 3])  # Global Average Pool → [B, 128]
        return self.proj(x)      # [B, d_model]

cnn = TinyCNN(d_model=128)
cnn_params = sum(p.numel() for p in cnn.parameters())

# Effective receptive field of 3-layer CNN
# Layer 1: 5×5 kernel, stride 2 → RF = 5
# Layer 2: 3×3, stride 2 → RF = 5 + (3-1)*2 = 9
# Layer 3: 3×3, stride 2 → RF = 9 + (3-1)*4 = 17
# With stride accumulation: effective RF ≈ 17×17 pixels out of 224

print(f"Architecture Comparison:")
print(f"{'':>25s} {'TinyCNN':>15s} {'ViT-Tiny':>15s}")
print(f"-" * 58)
print(f"{'Parameters':>25s} {cnn_params:>15,} {total_params:>15,}")
print(f"{'Output shape':>25s} {'[B, 128]':>15s} {'[B, 197, 384]':>15s}")
print(f"{'Spatial info preserved':>25s} {'No (pooled)':>15s} {'Yes (per-patch)':>15s}")
print(f"{'Receptive field':>25s} {'~17×17 pixels':>15s} {'224×224 (full)':>15s}")
print(f"{'Cross-patch reasoning':>25s} {'Local only':>15s} {'Global (attn)':>15s}")

In [ ]:
# Visualize: CNN receptive field vs ViT attention field
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: CNN — local receptive field
ax = axes[0]
ax.imshow(scene, alpha=0.3)
ax.imshow(scene)
# Show the limited receptive field: a 17×17 pixel area
# For a neuron looking at the cup area
rf_size = 37  # effective RF after 3 layers with strides
cx, cy = 56, 75  # cup center
# Darken everything outside RF
overlay = np.ones_like(scene) * 0.15
y0 = max(0, cy - rf_size//2)
y1 = min(224, cy + rf_size//2)
x0 = max(0, cx - rf_size//2)
x1 = min(224, cx + rf_size//2)
overlay[y0:y1, x0:x1] = 1.0
ax.imshow(scene * overlay)
rect = Rectangle((x0, y0), x1-x0, y1-y0,
                  linewidth=3, edgecolor=ACCENT, facecolor='none', linestyle='--')
ax.add_patch(rect)
ax.set_title(f'CNN: Local Receptive Field (~{rf_size}×{rf_size} px)',
             fontsize=13, color=ACCENT, fontweight='bold')
ax.text(112, 210, 'Cannot see gripper, plate, or table edge!',
        fontsize=10, ha='center', color=ACCENT, fontweight='bold')
ax.axis('off')

# Right: ViT — global attention
ax = axes[1]
ax.imshow(scene)
# Show attention arrows from cup patch to distant patches
cup_center = (56, 75)
targets = [
    (168, 75, 'Blue plate', BLUE),
    (134, 35, 'Gripper', 'gray'),
    (112, 112, 'Green cube', TEAL),
    (112, 180, 'Table edge', WARM),
]
for tx, ty, label, c in targets:
    ax.annotate('', xy=(tx, ty), xytext=cup_center,
                arrowprops=dict(arrowstyle='->', color=c, lw=2.5))
    ax.text(tx, ty + 12, label, fontsize=9, ha='center', color=c, fontweight='bold')

# Highlight source patch
rect = Rectangle((cx - PATCH_SIZE//2, cy - PATCH_SIZE//2),
                  PATCH_SIZE, PATCH_SIZE,
                  linewidth=3, edgecolor=ACCENT, facecolor='none')
ax.add_patch(rect)
ax.set_title('ViT: Global Attention (sees everything)',
             fontsize=13, color=TEAL, fontweight='bold')
ax.text(112, 210, 'Cup attends to gripper, plate, table — instant global reasoning!',
        fontsize=10, ha='center', color=TEAL, fontweight='bold')
ax.axis('off')

plt.suptitle('CNN vs ViT: What Can the Cup "See"?',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("CNN: The cup neuron can only see a tiny local neighborhood.")
print("  It has NO IDEA where the gripper is or what color the plate is.")
print("  It takes many layers (and strides) to eventually 'see' the whole image.")
print("\nViT: The cup patch attends to the ENTIRE image in a single layer.")
print("  It knows the gripper is approaching, the plate is to the right, etc.")
print("  This is why ViT replaced CNNs for robot vision.")

---
## Step 7: The Projection Bridge

The ViT outputs d_v-dimensional tokens (e.g. 768-d for ViT-Base). The language model (e.g. Gemma) expects d_model-dimensional tokens (e.g. 2048-d). A **single linear projection** bridges the gap.

In [ ]:
# Simulate the projection bridge
D_VIT = 768        # ViT-Base output dimension
D_LLM = 2048       # LLM embedding dimension (e.g. Gemma)
N_VISUAL_TOKENS = 196  # 14×14 patches

projection = nn.Linear(D_VIT, D_LLM)

# Simulate ViT output
vit_tokens = torch.randn(1, N_VISUAL_TOKENS, D_VIT)

with torch.no_grad():
    llm_tokens = projection(vit_tokens)  # [1, 196, 2048]

proj_params = sum(p.numel() for p in projection.parameters())

print(f"Projection Bridge:")
print(f"  ViT output: {list(vit_tokens.shape)} ({D_VIT}-d per token)")
print(f"  After projection: {list(llm_tokens.shape)} ({D_LLM}-d per token)")
print(f"  Parameters: {proj_params:,} ({D_VIT}×{D_LLM} + {D_LLM} bias)")
print(f"\nThat's it — just a matrix multiply + bias.")
print(f"Why not a deep MLP? The ViT's features are already rich.")
print(f"A linear map just 'translates' them into the LLM's coordinate system.")

---
## Step 8: VLM Assembly — Visual + Text Tokens in One Sequence

This is the magic of Vision-Language Models: **concatenate** visual tokens and text tokens into one sequence, then let the Transformer's self-attention do cross-modal reasoning.

In [ ]:
# Simulate VLM assembly: PaliGemma-style
N_TEXT_TOKENS = 7  # "pick up the red cup from table"
text_words = ["pick", "up", "the", "red", "cup", "from", "table"]

# Simulated visual tokens (from ViT + projection)
visual_tokens = torch.randn(1, N_VISUAL_TOKENS, D_LLM)  # [1, 196, 2048]

# Simulated text tokens (from tokenizer + embedding)
text_tokens = torch.randn(1, N_TEXT_TOKENS, D_LLM)  # [1, 7, 2048]

# Concatenate: [visual tokens | text tokens]
vlm_input = torch.cat([visual_tokens, text_tokens], dim=1)  # [1, 203, 2048]

print(f"VLM Input Assembly:")
print(f"  Visual tokens: {list(visual_tokens.shape)} (196 image patches)")
print(f"  Text tokens:   {list(text_tokens.shape)} ({N_TEXT_TOKENS} words)")
print(f"  Combined:      {list(vlm_input.shape)} (one unified sequence)")
print(f"\nSequence layout:")
print(f"  Positions 0-195:   Image patches (14×14 grid)")
print(f"  Positions 196-202: Text tokens ({' | '.join(text_words)})")
print(f"\nNow self-attention connects EVERYTHING:")
print(f"  'red' (text) ↔ red-colored patches (image)")
print(f"  'cup' (text) ↔ cup-shaped patches (image)")
print(f"  'pick' (text) ↔ gripper patches (image)")

In [ ]:
# Visualize the VLM attention pattern
N_TOTAL = N_VISUAL_TOKENS + N_TEXT_TOKENS  # 203

# Simulate a trained VLM's cross-modal attention
# In reality this emerges from training — we'll create a plausible pattern
torch.manual_seed(123)
fake_attn = torch.rand(N_TOTAL, N_TOTAL) * 0.02  # low baseline

# Visual tokens attend mostly to nearby visual tokens
for i in range(N_VISUAL_TOKENS):
    row_i = i // 14
    col_i = i % 14
    for j in range(N_VISUAL_TOKENS):
        row_j = j // 14
        col_j = j % 14
        dist = abs(row_i - row_j) + abs(col_i - col_j)
        if dist <= 2:
            fake_attn[i, j] += 0.15

# "red" (index 199) attends strongly to patches in the red cup area (rows 3-5, cols 2-4)
red_idx = N_VISUAL_TOKENS + 3  # "red"
cup_idx = N_VISUAL_TOKENS + 4  # "cup"
pick_idx = N_VISUAL_TOKENS + 0  # "pick"

for r in range(3, 6):
    for c in range(2, 5):
        fake_attn[red_idx, r * 14 + c] += 0.3  # red → red patches
        fake_attn[cup_idx, r * 14 + c] += 0.25  # cup → cup patches

# "pick" attends to gripper patches (row 0-3, col 8-9)
for r in range(0, 4):
    for c in range(8, 10):
        fake_attn[pick_idx, r * 14 + c] += 0.25

# Normalize rows
fake_attn = F.softmax(fake_attn * 5, dim=-1).numpy()

# Show cross-modal attention for "red", "cup", "pick" as maps over the image
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, word, tok_idx, c in [
    (axes[0], '"pick"', pick_idx, ACCENT),
    (axes[1], '"red"', red_idx, BLUE),
    (axes[2], '"cup"', cup_idx, TEAL),
]:
    # Get attention from this text token to all image patches
    text_to_image = fake_attn[tok_idx, :N_VISUAL_TOKENS]  # [196]
    attn_map = text_to_image.reshape(14, 14)

    ax.imshow(scene)
    ax.imshow(np.kron(attn_map, np.ones((PATCH_SIZE, PATCH_SIZE))),
              alpha=0.65, cmap='hot', vmin=0)
    ax.set_title(f'{word} → image patches', fontsize=13, color=c, fontweight='bold')
    ax.axis('off')

plt.suptitle('Cross-Modal Attention: Text Words Attend to Image Regions (simulated)',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print('"pick" attends to the GRIPPER — it knows which part of the scene does the picking.')
print('"red" attends to RED PIXELS — it grounds the color word to visual features.')
print('"cup" attends to CUP PATCHES — it knows what the target object looks like.')
print("\nNo one programmed these patterns!")
print("They EMERGE from next-token prediction on millions of image-text pairs.")
print("This is exactly what our mini-VLA's MLP fusion could never do.")

---
## Step 9: Putting It All Together — The VLA Pipeline

Modern VLAs (RT-2, OpenVLA, pi0) chain these components:

```
Camera Image → ViT (vision encoder) → Projection → [visual tokens]
                                                          ↓
Text Instruction → Tokenizer → Embedding → [text tokens]
                                                          ↓
                                     [visual | text] → Transformer (LLM) → Action Tokens
```

In [ ]:
# Summary: parameter count comparison
components = [
    ("Our TinyCNN (Part 2)", 128, "200 PushT frames", "~50K"),
    ("Our ViT-Tiny (this notebook)", f"{total_params/1e6:.1f}M", "N/A (random init)", "~7M"),
    ("ViT-Base (standard)", "86M", "ImageNet (or similar)", "86M"),
    ("Gemma 2B (LLM)", "2B", "Trillions of tokens", "2B"),
    ("Full VLM (ViT + LLM)", "~2.4B", "Internet-scale", "~2.4B"),
]

print(f"{'Component':>35s} | {'Params':>10s} | {'Training Data':>25s}")
print("-" * 80)
for name, params, data, _ in components:
    print(f"{name:>35s} | {str(params):>10s} | {data:>25s}")

print(f"\n{'='*80}")
print(f"The mini-VLA trained EVERYTHING from scratch on 200 demos.")
print(f"A modern VLA plugs in BILLIONS of parameters of pre-trained knowledge")
print(f"and only fine-tunes the action head on robot data.")
print(f"\nThat's why it generalizes to new objects, new instructions, and new scenes.")

---
## Summary

| Component | What it does | Why it matters for robots |
|-----------|-------------|---------------------------|
| **Patch Embedding** | Image → 196 patch tokens | Transforms images into a Transformer-compatible sequence |
| **ViT Self-Attention** | Every patch sees every patch | Global reasoning: cup ↔ gripper ↔ table in one layer |
| **CLS Token** | Aggregates all patch info | Single image-level vector when you need one |
| **Projection Bridge** | ViT d_v → LLM d_model | Connects vision encoder to language model |
| **VLM Assembly** | [visual tokens \| text tokens] | Cross-modal attention: "red" → red patches |

### The Full Modern VLA Recipe

1. **ViT** encodes the camera image into visual tokens
2. **Linear projection** translates visual tokens into the LLM's embedding space
3. **LLM** processes [visual + text] tokens with cross-modal attention
4. **Action head** (fine-tuned on robot data) generates motor commands

All the "world knowledge" comes for free from pre-training. The robot only needs to learn the last mile: turning understanding into action.

---

## Exercises

1. **Change patch size:** Try 8×8 and 32×32 patches. How does the number of tokens change? What are the trade-offs (more tokens = finer detail but O(N²) attention cost)?
2. **Visualize your own image:** Load a real photo with `PIL.Image`, resize to 224×224, and run it through our ViT. Visualize the CLS attention map — which regions does the model focus on?
3. **Pre-trained ViT:** Load a real pre-trained ViT from torchvision (`torchvision.models.vit_b_16(pretrained=True)`) and compare its attention maps to our random ViT. The difference shows what training buys you.
4. **Challenge:** Combine this ViT with the Transformer from `Transformer_From_Scratch.ipynb` to build a mini VLM. Feed image patches + text tokens into a shared Transformer and visualize the cross-modal attention.